# Silver: chamados_hora

## Padronização
- Nomes de coluna em minúsculo.
- `id_chamado`: convertido com try_cast para bigint (mesmo padrão do chamados). `id_cliente` continua texto, para não perder zero à esquerda.
- Horários: "às"/"as" removido e convertido de DD/MM/YYYY HH:MM:SS para timestamp.

## Regras de limpeza
1. `id_chamado` inválido ou `id_cliente` ausente: remove a linha.
2. Hora de abertura ausente ou inválida: remove.
3. Ordem temporal inválida (início antes da abertura, fim antes do início ou da abertura): remove.
4. Início ou fim preenchidos mas inválidos: o campo vira nulo.
5. ID repetido: cópias idênticas mantêm uma; dados diferentes anulam todas.

## Decisões
- Linhas sem hora de início **não são removidas**: ficam marcadas em `sem_inicio_atendimento` (podem ser abandonos).
- A validação cruzada **marca e não remove**: `cliente_valido` (existe em silver.clientes) e `chamado_valido` (existe em silver.chamados). O clientes remove idades fora de 18 a 110, e apagar essas linhas aqui tiraria chamados de clientes reais.

In [0]:
from pyspark.sql import functions as F, Window

catalog = "rocket_lab"
bronze_schema = f"{catalog}.vcredit_bronze"
silver_schema = f"{catalog}.vcredit_silver"
tabela_destino = f"{silver_schema}.chamados_hora"

HORAS = ["hora_abertura_chamado", "hora_inicio_atendimento", "hora_finalizacao_atendimento"]

In [0]:
df = spark.table(f"{bronze_schema}.chamados_hora")
df = df.toDF(*[c.lower() for c in df.columns])
n_bronze = df.count()

# ids
df = (df
    .withColumn("id_chamado", F.expr("try_cast(trim(id_chamado) as bigint)"))
    .withColumn("id_cliente", F.trim("id_cliente")))

# Horários: tira o "às"/"as" e converte; texto inválido vira nulo
for c in HORAS:
    bruto = F.trim(F.col(c))
    sem_as = F.regexp_replace(bruto, r"(?iu)[\s\u00A0]*[aà]s[\s\u00A0]*", " ")
    preenchido = bruto.isNotNull() & ~F.upper(bruto).isin("", "NULL", "NAN", "NONE")
    df = df.withColumn(f"{c}_txt", F.when(preenchido, sem_as))
    df = df.withColumn(c, F.expr(f"try_to_timestamp({c}_txt, 'dd/MM/yyyy HH:mm:ss')"))

ab = F.col("hora_abertura_chamado")
ini = F.col("hora_inicio_atendimento")
fim = F.col("hora_finalizacao_atendimento")
def sim(cond): return F.coalesce(cond, F.lit(False))

# Marca cada regra em uma coluna
df = (df
    .withColumn("r_chave_ausente",
        F.col("id_chamado").isNull() | F.col("id_cliente").isNull() | (F.col("id_cliente") == ""))
    .withColumn("r_abertura_invalida", ab.isNull())
    .withColumn("r_inicio_antes_abertura", sim(ini < ab))
    .withColumn("r_fim_antes_inicio", sim(fim < ini))
    .withColumn("r_fim_antes_abertura", sim(fim < ab))
    .withColumn("r_inicio_texto_invalido", F.col("hora_inicio_atendimento_txt").isNotNull() & ini.isNull())
    .withColumn("r_fim_texto_invalido", F.col("hora_finalizacao_atendimento_txt").isNotNull() & fim.isNull())
    .withColumn("sem_inicio_atendimento", ini.isNull())
)

# Contagem de cada regra (uma linha pode cair em mais de uma)
regras = {
    "r_chave_ausente":         "REMOVE: id_chamado inválido ou id_cliente ausente",
    "r_abertura_invalida":     "REMOVE: hora de abertura ausente ou inválida",
    "r_inicio_antes_abertura": "REMOVE: início antes da abertura",
    "r_fim_antes_inicio":      "REMOVE: fim antes do início",
    "r_fim_antes_abertura":    "REMOVE: fim antes da abertura",
    "r_inicio_texto_invalido": "MANTÉM (nulo): início preenchido mas inválido",
    "r_fim_texto_invalido":    "MANTÉM (nulo): fim preenchido mas inválido",
    "sem_inicio_atendimento":  "MANTÉM e sinaliza: sem hora de início",
}
cont = df.agg(*[F.sum(F.col(r).cast("int")).alias(r) for r in regras]).first().asDict()

remover = (F.col("r_chave_ausente") | F.col("r_abertura_invalida")
           | F.col("r_inicio_antes_abertura") | F.col("r_fim_antes_inicio")
           | F.col("r_fim_antes_abertura"))
df = df.filter(~remover)

# Duplicados: cópias idênticas mantêm uma; mesmo id_chamado com dados diferentes anula todas
n_antes = df.count()
df = df.dropDuplicates(["id_chamado", "id_cliente"] + HORAS)
n_identicas = n_antes - df.count()

w = Window.partitionBy("id_chamado")
df = df.withColumn("qtd", F.count("*").over(w))
n_conflito = df.filter("qtd > 1").count()
df = df.filter("qtd = 1").drop("qtd")

# Validação cruzada: MARCA, não remove
cli_ids = (spark.table(f"{silver_schema}.clientes").select("id_cliente")
           .where(F.col("id_cliente").isNotNull()).distinct()
           .withColumn("cliente_valido", F.lit(True)))
cha_ids = (spark.table(f"{silver_schema}.chamados").select("id_chamado")
           .where(F.col("id_chamado").isNotNull()).distinct()
           .withColumn("chamado_valido", F.lit(True)))

df = (df.join(cli_ids, "id_cliente", "left")
        .join(cha_ids, "id_chamado", "left")
        .withColumn("cliente_valido", F.coalesce("cliente_valido", F.lit(False)))
        .withColumn("chamado_valido", F.coalesce("chamado_valido", F.lit(False))))

df = df.select("id_chamado", "id_cliente", *HORAS,
               "sem_inicio_atendimento", "cliente_valido", "chamado_valido",
               "ingest_timestamp")

# Gravação (lê da bronze e sobrescreve)
(df.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(tabela_destino))

# Resumo
t = spark.table(tabela_destino)
n_silver = t.count()
print(f"bronze: {n_bronze} linhas")
for r, descricao in regras.items():
    print(f"  {descricao}: {cont[r]}")
print(f"  cópias idênticas removidas: {n_identicas}")
print(f"  linhas anuladas por id_chamado conflitante: {n_conflito}")
print(f"  MARCADAS: cliente não encontrado em clientes: {t.filter(~F.col('cliente_valido')).count()}")
print(f"  MARCADAS: chamado não encontrado em chamados: {t.filter(~F.col('chamado_valido')).count()}")
print(f"silver: {n_silver} linhas")

display(t.agg(F.min("hora_abertura_chamado"), F.max("hora_abertura_chamado")))  # deve ficar entre jan/2025 e jul/2026
display(t.limit(20))